# Train Spacenet UNet (self-contained)

This notebook defines the model architecture in-place (no backend imports) and saves weights to the same path the API loads from.

Before running, update `DATA_DIR` and `FOLDS_CSV` to point at your SpaceNet data layout.

In [ ]:
from pathlib import Path
import os
import time

import torch

HERE = Path.cwd().resolve()
if (HERE / "backend").exists():
    PROJECT_ROOT = HERE
elif HERE.name == "backend":
    PROJECT_ROOT = HERE.parent
else:
    PROJECT_ROOT = HERE

BACKEND_DIR = PROJECT_ROOT / "backend"
WEIGHTS_OUT_PATH = BACKEND_DIR / "weights" / "spacenet_irv_unet_inceptionresnetv2_0_best_dice"

# Update these to match your dataset layout.
DATA_DIR = PROJECT_ROOT / "Dataset" / "cresi_train"
FOLDS_CSV = DATA_DIR / "folds.csv"

IMAGE_TYPE = "PS-RGB"
NUM_CLASSES = 12
INPUT_CHANNELS = 3
CROP_HEIGHT = 384
CROP_WIDTH = 384
BATCH_SIZE = 10
EPOCHS = 38
LEARNING_RATE = 1e-4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

In [ ]:
import cv2
import numpy as np
import gdal
import pandas as pd
import skimage.io
import torch.nn as nn
import torch.nn.functional as F

from albumentations import (
    Compose, RandomSizedCrop, HorizontalFlip, VerticalFlip, RGBShift,
    RandomBrightnessContrast, RandomGamma, OneOf, RandomRotate90,
    PadIfNeeded, Transpose, RandomCrop, Rotate, ShiftScaleRotate
)
from albumentations.pytorch.transforms import img_to_tensor
from torch.utils.data import Dataset, DataLoader

torch.backends.cudnn.benchmark = True
cv2.ocl.setUseOpenCL(False)
cv2.setNumThreads(0)

In [ ]:
cities = ["Vegas", "Paris", "Shanghai", "Khartoum", "Moscow", "Mumbai"]

def get_city_idx(file_name):
    for i, city in enumerate(cities):
        if city in file_name:
            return i
    raise ValueError(f"unknown city for {file_name}")

class SpacenetSimpleDataset(Dataset):
    def __init__(self, data_path, mode, fold=0, folds_csv="folds.csv", transforms=None, normalize=None, multiplier=1,
                 image_type="PS-RGB"):
        super().__init__()
        self.data_path = data_path
        self.mode = mode
        self.image_type = image_type
        self.names = sorted(os.listdir(os.path.join(self.data_path, image_type)))
        df = pd.read_csv(folds_csv)
        self.df = df
        self.normalize = normalize
        self.fold = fold
        if self.mode == "train":
            ids = set(df[df["fold"] != fold]["id"].tolist())
        else:
            ids = set(df[df["fold"] == fold]["id"].tolist())
        self.names = [n for n in self.names if n[:-4].replace("-MS", "-RGB") in ids]
        if mode == "val":
            self.names = [n for n in self.names if "Moscow" in n or "Mumbai" in n or "Vegas" in n]
        self.transforms = transforms
        if mode == "train":
            self.names = self.names * multiplier

    def __len__(self):
        return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        img_path = os.path.join(self.data_path, self.image_type, name)
        image = skimage.io.imread(img_path)
        if image is None:
            raise ValueError(img_path)
        image_path = os.path.join(self.data_path, "train_mask_binned_mc_10", name[:-4].replace("RGB", "MS") + ".tif")
        try:
            raster_ds = gdal.Open(image_path, gdal.GA_ReadOnly)
            speed_mask = raster_ds.ReadAsArray()
        except Exception:
            raster_ds = gdal.Open(image_path, gdal.GA_ReadOnly)
            speed_mask = raster_ds.ReadAsArray()
        if speed_mask.shape[-1] > 100:
            speed_mask = np.moveaxis(speed_mask, 0, -1)
        junction_mask = cv2.imread(
            os.path.join(self.data_path, "junction_masks", name[:-4].replace("PS-RGB_", "").replace("PS-MS_", "") + ".png"),
            cv2.IMREAD_GRAYSCALE,
        )
        if junction_mask is None:
            junction_mask = np.zeros((1300, 1300))
        if speed_mask is None:
            speed_mask = np.zeros((1300, 1300, 11), dtype=np.uint8)
        sample = self.transforms(
            image=image,
            mask=np.concatenate([speed_mask, np.expand_dims(junction_mask, -1), np.expand_dims(junction_mask, -1)], axis=-1),
            img_name=name,
        )
        mask = sample["mask"]
        city = np.zeros((len(cities), 1, 1))
        city[get_city_idx(name), 0, 0] = 1
        sample["city"] = city
        sample["img_name"] = name
        mask = np.moveaxis(mask, -1, 0) / 255.0
        sample["mask"] = torch.from_numpy(mask)
        sample["image"] = img_to_tensor(sample["image"], self.normalize)
        return sample

def create_train_transforms():
    return Compose([
        ShiftScaleRotate(shift_limit=0.2, scale_limit=0, rotate_limit=0),
        OneOf([
            RandomSizedCrop(min_max_height=(int(CROP_HEIGHT * 0.8), int(CROP_HEIGHT * 1.2)),
                            w2h_ratio=1., height=CROP_HEIGHT, width=CROP_WIDTH, p=0.9),
            RandomCrop(height=CROP_HEIGHT, width=CROP_WIDTH, p=0.1)
        ], p=1),
        Rotate(limit=10, p=0.2, border_mode=cv2.BORDER_CONSTANT, value=0),
        HorizontalFlip(),
        VerticalFlip(),
        RandomRotate90(),
        Transpose(),
        OneOf([RGBShift(), RandomBrightnessContrast(), RandomGamma()], p=0.5),
    ])

def create_val_transforms():
    return Compose([
        PadIfNeeded(min_height=1344, min_width=1344),
    ])

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.block = ConvBlock(in_ch, out_ch)

    def forward(self, x):
        return self.block(self.pool(x))

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
        self.block = ConvBlock(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        if x1.shape[-2:] != x2.shape[-2:]:
            diff_y = x2.size(2) - x1.size(2)
            diff_x = x2.size(3) - x1.size(3)
            x1 = F.pad(x1, [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.block(x)

class UNet(nn.Module):
    def __init__(self, in_channels, num_classes, base_filters=32):
        super().__init__()
        self.inc = ConvBlock(in_channels, base_filters)
        self.down1 = Down(base_filters, base_filters * 2)
        self.down2 = Down(base_filters * 2, base_filters * 4)
        self.down3 = Down(base_filters * 4, base_filters * 8)
        self.down4 = Down(base_filters * 8, base_filters * 16)
        self.up1 = Up(base_filters * 16 + base_filters * 8, base_filters * 8)
        self.up2 = Up(base_filters * 8 + base_filters * 4, base_filters * 4)
        self.up3 = Up(base_filters * 4 + base_filters * 2, base_filters * 2)
        self.up4 = Up(base_filters * 2 + base_filters, base_filters)
        self.outc = nn.Conv2d(base_filters, num_classes, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

def dice_loss(probs, targets, eps=1e-5):
    targets = targets.float()
    probs = probs.float()
    num = 2 * torch.sum(probs * targets, dim=(2, 3)) + eps
    den = torch.sum(probs + targets, dim=(2, 3)) + eps
    loss = 1 - (num / den)
    return loss.mean()

bce_logits = nn.BCEWithLogitsLoss()

def compute_loss(logits, targets):
    mask_band = 10
    jn_band = 11
    speed_loss = bce_logits(logits[:, :mask_band, ...], targets[:, :mask_band, ...])
    mask_logits = logits[:, mask_band:jn_band, ...]
    mask_targets = targets[:, mask_band:jn_band, ...]
    mask_bce = bce_logits(mask_logits, mask_targets)
    mask_dice = dice_loss(torch.sigmoid(mask_logits), mask_targets)
    junction_loss = bce_logits(logits[:, jn_band:jn_band + 1, ...], targets[:, jn_band:jn_band + 1, ...])
    total = speed_loss + mask_bce + mask_dice + junction_loss
    return total, mask_dice

model = UNet(INPUT_CHANNELS, NUM_CLASSES, base_filters=32).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("model ready")

In [ ]:
train_dataset = SpacenetSimpleDataset(
    mode="train",
    fold=0,
    image_type=IMAGE_TYPE,
    data_path=str(DATA_DIR),
    folds_csv=str(FOLDS_CSV),
    transforms=create_train_transforms(),
    multiplier=1,
    normalize={"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]},
)
val_dataset = SpacenetSimpleDataset(
    mode="val",
    fold=0,
    image_type=IMAGE_TYPE,
    data_path=str(DATA_DIR),
    folds_csv=str(FOLDS_CSV),
    transforms=create_val_transforms(),
    normalize={"mean": [0.5, 0.5, 0.5], "std": [0.5, 0.5, 0.5]},
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))

In [ ]:
def train_one_epoch(model, loader, optimizer, epoch, use_amp=False):
    model.train()
    losses_meter = []
    dices_meter = []
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    for i, sample in enumerate(loader):
        imgs = sample["image"].to(DEVICE)
        masks = sample["mask"].to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)
            loss, mask_dice = compute_loss(logits, masks)
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        losses_meter.append(loss.item())
        dices_meter.append(1 - mask_dice.item())
    avg_loss = float(np.mean(losses_meter)) if losses_meter else 0.0
    avg_dice = float(np.mean(dices_meter)) if dices_meter else 0.0
    return avg_loss, avg_dice

def validate_one_epoch(model, loader):
    model.eval()
    dices = []
    with torch.no_grad():
        for sample in loader:
            imgs = sample["image"].to(DEVICE)
            masks = sample["mask"].to(DEVICE)
            logits = model(imgs)
            _, mask_dice = compute_loss(logits, masks)
            dices.append(1 - mask_dice.item())
    return float(np.mean(dices)) if dices else 0.0

In [ ]:
use_amp = bool(torch.cuda.is_available())

best_dice = -1.0
WEIGHTS_OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(EPOCHS):
    start = time.time()
    train_loss, train_dice = train_one_epoch(model, train_loader, optimizer, epoch, use_amp=use_amp)
    val_dice = validate_one_epoch(model, val_loader)
    elapsed = time.time() - start
    print(f"epoch {epoch + 1}/{EPOCHS} loss={train_loss:.4f} train_dice={train_dice:.4f} val_dice={val_dice:.4f} time={elapsed:.1f}s")

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(
            {
                "epoch": epoch + 1,
                "state_dict": model.state_dict(),
                "dice_best": best_dice,
            },
            str(WEIGHTS_OUT_PATH),
        )
        print(f"saved best dice checkpoint to {WEIGHTS_OUT_PATH}")